In [0]:
%pip install -q pdfplumber
dbutils.library.restartPython()


In [0]:
%restart_python

In [0]:
import os
from pathlib import Path
import pandas as pd
from pyspark.sql import functions as F, types as T

# Mismo catálogo que el notebook Lector
CATALOGO          = "equipo4"
CATALOGO_RESPALDO = "workspace"

try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
except Exception as e:
    print(f"No pude crear el catálogo '{CATALOGO}' ({str(e).splitlines()[0]}). Uso '{CATALOGO_RESPALDO}'.")
    CATALOGO = CATALOGO_RESPALDO
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.bronze.archivos")

CARPETA_MAGA_UC   = Path(f"/Volumes/{CATALOGO}/bronze/archivos/maga")
T_BRONZE_MAGA_PDF = f"{CATALOGO}.bronze.maga_anuarios_pdf"
T_BRONZE_MAGA     = f"{CATALOGO}.bronze.precios_maga_raw"


In [0]:
import io, re, unicodedata
import pdfplumber

ANIOS_MAGA = [2024, 2025]     # agrega años si los necesitas; cada anuario trae varios años hacia atrás
URL_MAGA   = "https://precios.maga.gob.gt/archivos/anuarios/anuario-{anio}.pdf"

def _sin_acentos(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()

_PRODUCTO = re.compile(r"^(gasolina\s+super|gasolina\s+regular|diesel|super|regular)\b\s*(.*)$")
_ANIO     = re.compile(r"^(20\d{2})\s+(.*)$")
_NUM      = re.compile(r"\d+(?:[.,]\d+)?")

def extraer_de_texto(texto, pagina=None):
    """Filas del Cuadro 1.19: 'Diésel' / 'Gasolina Súper' / 'Gasolina Regular' seguidas de renglones
    'AAAA ene feb ... dic promedio'. Solo se aceptan renglones con 13 números (12 meses + promedio)."""
    filas, producto, avisos = [], None, []
    for linea in texto.splitlines():
        l = _sin_acentos(linea).strip()
        m = _PRODUCTO.match(l)
        if m:
            producto = m.group(1).split()[-1]          # diesel / super / regular
            l = m.group(2)
            if not l:
                continue
        m = _ANIO.match(l)
        if not (producto and m):
            continue
        nums = _NUM.findall(m.group(2))
        if len(nums) != 13:
            avisos.append(f"pág {pagina}: renglón raro ({producto} {m.group(1)}): {linea.strip()}")
            continue
        for mes, v in enumerate(nums[:12], start=1):
            filas.append({"combustible": producto, "anio": int(m.group(1)), "mes": mes,
                          "precio_q_galon": float(v.replace(",", ".")),
                          "promedio_anual_pdf": float(nums[12].replace(",", ".")), "pagina": pagina})
    return filas, avisos

def extraer_cuadro_combustibles(contenido_pdf):
    """Busca la(s) página(s) del cuadro de combustibles de la Ciudad Capital y extrae sus filas."""
    filas, avisos = [], []
    with pdfplumber.open(io.BytesIO(contenido_pdf)) as pdf:
        for num, pag in enumerate(pdf.pages, start=1):
            texto = pag.extract_text() or ""
            t = _sin_acentos(texto)
            if "combustibles" in t and "precios promedio mensuales" in t and "ciudad capital" in t:
                f, a = extraer_de_texto(texto, num)
                filas += f; avisos += a
    return filas, avisos


In [0]:
import urllib.request

# 1) Conseguir los PDF: descarga directa; si falla, los que estén subidos al volumen
pdfs = {}
for anio in ANIOS_MAGA:
    url = URL_MAGA.format(anio=anio)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=90) as r:
            pdfs[f"anuario-{anio}.pdf"] = (url, r.read())
        print(f"Descargado {url}")
    except Exception as e:
        print(f"No pude descargar {url}: {e}")
if CARPETA_MAGA_UC.is_dir():
    for f in sorted(CARPETA_MAGA_UC.glob("*.pdf")):
        if f.name not in pdfs:
            pdfs[f.name] = (str(f), f.read_bytes())
            print(f"Tomado del volumen: {f}")
if not pdfs:
    print(f"⚠️  Sin PDF del MAGA. Súbelos a {CARPETA_MAGA_UC} y vuelve a correr esta celda.")

ahora = pd.Timestamp.now().to_pydatetime()



In [0]:
# 2) BRONZE: los PDF tal cual (MERGE por nombre de archivo: no se duplican)
if pdfs:
    esquema_pdf = T.StructType([T.StructField("archivo", T.StringType()), T.StructField("origen", T.StringType()),
                                T.StructField("bytes", T.LongType()), T.StructField("contenido", T.BinaryType()),
                                T.StructField("ingestado_en", T.TimestampType())])
    spark.createDataFrame([(n, u, len(b), bytearray(b), ahora) for n, (u, b) in pdfs.items()],
                          esquema_pdf).createOrReplaceTempView("_maga_pdf")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {T_BRONZE_MAGA_PDF} (archivo STRING, origen STRING, bytes BIGINT, "
              "contenido BINARY, ingestado_en TIMESTAMP)")
    spark.sql(f"""MERGE INTO {T_BRONZE_MAGA_PDF} t USING _maga_pdf s ON t.archivo = s.archivo
                  WHEN MATCHED AND s.bytes <> t.bytes THEN UPDATE SET *
                  WHEN NOT MATCHED THEN INSERT *""")



In [0]:
# 3) BRONZE: la tabla extraída de cada PDF guardado (sin limpiar ni quitar años repetidos)
if spark.catalog.tableExists(T_BRONZE_MAGA_PDF):
    filas = []
    for r in spark.table(T_BRONZE_MAGA_PDF).select("archivo", "contenido").toLocalIterator():
        f, avisos = extraer_cuadro_combustibles(bytes(r.contenido))
        for a in avisos:
            print(f"{r.archivo}: {a}")
        print(f"{r.archivo}: {len(f)} valores extraídos" + ("" if f else "  ⚠️ no encontré el cuadro"))
        filas += [{**x, "archivo_fuente": r.archivo} for x in f]
    if filas:
        maga = pd.DataFrame(filas)
        maga["extraido_en"] = pd.Timestamp.now()
        spark.createDataFrame(maga).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T_BRONZE_MAGA)
        print(f"Bronze: {T_BRONZE_MAGA}  ({len(maga)} filas)")
        display(spark.table(T_BRONZE_MAGA).groupBy("archivo_fuente", "combustible")
                .agg(F.min("anio").alias("desde"), F.max("anio").alias("hasta"), F.count("*").alias("valores")))
